# MACELES-OFF: precision check + periodic bulk water

Two follow-ups to the water-dimer scan from the quickstart notebook:

1. **Precision check** — the far-field tail (10–20 Å) looked suspiciously flat at `float32`. Rerun it at `float64` to see whether that's a real slowly-decaying tail or a rounding floor.
2. **Periodic bulk water** — the isolated dimer only exercises LES's *pairwise/direct-space* branch. A periodic box exercises the actual reciprocal-space Ewald summation, which is where the MACELES-OFF paper reported its biggest gains (liquid properties, IR spectra).

This notebook is self-contained — it re-does the install and checkpoint loading from the quickstart notebook, so you can run it standalone in a fresh Colab session.

In [ ]:
!pip install -q ase numpy
!pip install -q "git+https://github.com/ACEsuit/mace.git"
!pip install -q "git+https://github.com/ChengUCB/les.git"

In [ ]:
import os
import glob
import subprocess

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

repo_dir = "les_fit"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/ChengUCB/les_fit.git"], check=True)

candidates = sorted(glob.glob(f"{repo_dir}/**/*.model", recursive=True))
off_candidates = [c for c in candidates if "off" in c.lower()]
converted = [c for c in off_candidates if "convert" in c.lower()]
model_path = (converted or off_candidates)[0]
print("Using checkpoint:", model_path)

In [ ]:
from mace.calculators import MACECalculator
from ase import build

def water_dimer(separation):
    a = build.molecule("H2O")
    b = build.molecule("H2O")
    b.translate([separation, 0.0, 0.0])
    return a + b

def monomer_energy(calc):
    w = build.molecule("H2O")
    w.calc = calc
    return w.get_potential_energy()

## Part A — is the far-field tail real, or is it float32 rounding?

Same water-dimer scan as before, restricted to the range that looked flat (10–30 Å), now at `float64`.

In [ ]:
calc_les_f64 = MACECalculator(
    model_paths=model_path,
    device=device,
    default_dtype="float64",
    dispersion=False,
)

e_mono_f64 = monomer_energy(calc_les_f64)

distances_far = [10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 25.0, 30.0]

print(f"{'d (\u00c5)':>6} | {'E_int MACELES-OFF, float64 (meV)':>34}")
for d in distances_far:
    dimer = water_dimer(d)
    dimer.calc = calc_les_f64
    e_int = (dimer.get_potential_energy() - 2 * e_mono_f64) * 1000
    print(f"{d:6.1f} | {e_int:34.6f}")

**How to read this:** if the values now show a smooth, monotonically decreasing curve rather than repeating the same rounded number, the `float32` run was hitting its precision floor and the underlying tail is real (if weak). If the numbers are still essentially flat at `float64` too, the flatness is a property of the model/method itself at this level, not numerical noise — worth then checking whether `remove_self_interaction` or the Ewald k-space cutoff defaults are clipping the far-field contribution for a non-periodic (pairwise-summation) system.

## Part B — periodic bulk water

Build a small periodic box of water at roughly liquid density, relax it to remove bad contacts, then run brief NVT dynamics. Setting `pbc=True` and a `cell` on the `Atoms` object is what makes LES switch from pairwise direct-space summation to true reciprocal-space Ewald summation — no extra flags needed.

**Caveat:** this is a small, quickly-relaxed toy box (27 molecules, a few hundred optimizer/MD steps) built to demonstrate the periodic code path, not a rigorously equilibrated liquid-water simulation. For production liquid-water work you'd want a properly pre-equilibrated configuration (e.g. via Packmol + a longer NPT equilibration) and a much larger box.

In [ ]:
import numpy as np
from ase import Atoms

rng = np.random.default_rng(42)

n_side = 3
n_mol = n_side ** 3  # 27 waters

# Deliberately ~10% below true liquid density (0.0334 molecules/Å³) to give the
# coarse relaxation step more room to resolve the artificial grid contacts.
molecules_per_A3 = 0.030
volume = n_mol / molecules_per_A3
box_len = volume ** (1 / 3)
spacing = box_len / n_side
density_g_cm3 = molecules_per_A3 * 18.015 * 1.6605  # amu/Å³ -> g/cm³

print(f"{n_mol} waters, box = {box_len:.2f} Å cube, target density ≈ {density_g_cm3:.2f} g/cm³")

water_box = Atoms(cell=[box_len] * 3, pbc=True)

for i in range(n_side):
    for j in range(n_side):
        for k in range(n_side):
            mol = build.molecule("H2O")
            axis = rng.normal(size=3)
            axis /= np.linalg.norm(axis)
            angle = rng.uniform(0, 360)
            mol.rotate(angle, axis, center=mol.get_center_of_mass())
            target = (np.array([i, j, k]) + 0.5) * spacing
            mol.translate(target - mol.get_center_of_mass())
            water_box += mol

print("Total atoms:", len(water_box))

In [ ]:
from ase.optimize import FIRE

water_box.calc = calc_les_f64

print("Initial energy (eV):", water_box.get_potential_energy())

print("Stage 1: coarse relaxation to remove bad contacts...")
FIRE(water_box, logfile=None).run(fmax=1.0, steps=200)

print("Stage 2: tighter relaxation...")
FIRE(water_box, logfile=None).run(fmax=0.1, steps=300)

e_final = water_box.get_potential_energy()
print(f"Final energy (eV): {e_final:.3f}")
print(f"Final energy per molecule (eV): {e_final / n_mol:.3f}")

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

MaxwellBoltzmannDistribution(water_box, temperature_K=300)

dyn = Langevin(
    water_box,
    timestep=0.5 * units.fs,  # short timestep: box isn't fully equilibrated yet
    temperature_K=300,
    friction=0.05,
)

def log(a=water_box):
    epot = a.get_potential_energy()
    ekin = a.get_kinetic_energy()
    temp = ekin / (1.5 * units.kB * len(a))
    print(f"Epot={epot:10.3f} eV  Ekin={ekin:6.3f} eV  T={temp:6.1f} K")

dyn.attach(log, interval=20)

print("pbc:", water_box.pbc, " cell diag (\u00c5):", water_box.cell.diagonal())
print("Running 200 NVT steps under periodic boundary conditions...")
dyn.run(200)

## Notes

- `water_box.pbc == True` and a non-null `cell` are the only things that route LES through reciprocal-space Ewald summation instead of the direct pairwise sum used for the isolated dimer earlier — no separate calculator flag needed.
- If `Epot` in the MD log is still trending steeply downward rather than fluctuating around a plateau, the box hasn't equilibrated — that's expected here given the deliberately short relaxation; extend `steps` in the FIRE and Langevin cells if you want a more settled configuration.
- For anything you'd want to draw real physical conclusions from (radial distribution functions, dielectric constant, IR spectra), replace this toy box with a properly equilibrated one and run for much longer — this cell is here to demonstrate the periodic code path runs correctly, not to reproduce the paper's liquid-water results.